In [ ]:
import torch
import torch.nn as nn

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import time
import cv2
import numpy as np
import keyboard
import pyautogui
from IPython.display import clear_output

In [ ]:
num_hands        = 1
tracking_points  = 21
tracking_dim     = 3

latent_dim            = 16     # AE bottleneck — independent of num_action_classes
classifier_hidden_dim = 256
num_action_classes    = 4

run_device    = torch.device('cuda')
action_labels = ['a', 's', 'd', 'w']
press_buttons = False

In [ ]:
class CameraWrapper:
    def __init__(self, camera_index: int = 0):
        self.cap = cv2.VideoCapture(camera_index)
        if not self.cap.isOpened():
            raise IOError(f'Cannot open camera at index {camera_index}')

    def __call__(self) -> np.ndarray | None:
        ret, frame = self.cap.read()
        return frame if ret else None

    def release(self):
        if self.cap.isOpened():
            self.cap.release()

    def __enter__(self):  return self
    def __exit__(self, *_): self.release()

In [ ]:
class HandTracker:
    def __init__(self, model_path, max_num_hands=2,
                 min_hand_detection_confidence=0.5,
                 min_hand_presence_confidence=0.5,
                 min_tracking_confidence=0.5):
        base = python.BaseOptions(model_asset_path=model_path)
        self.options = vision.HandLandmarkerOptions(
            base_options=base,
            running_mode=vision.RunningMode.VIDEO,
            num_hands=max_num_hands,
            min_hand_detection_confidence=min_hand_detection_confidence,
            min_hand_presence_confidence=min_hand_presence_confidence,
            min_tracking_confidence=min_tracking_confidence,
        )
        self.landmarker = vision.HandLandmarker.create_from_options(self.options)

    def reset(self):
        self.landmarker.close()
        self.landmarker = vision.HandLandmarker.create_from_options(self.options)

    def __call__(self, bgr_frame: np.ndarray, timestamp_ms: int) -> torch.Tensor:
        rgb    = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
        result = self.landmarker.detect_for_video(
            mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb),
            timestamp_ms=timestamp_ms,
        )
        if not result.hand_landmarks:
            return torch.empty((0, 21, 3), dtype=torch.float32)
        out = np.zeros((len(result.hand_landmarks), 21, 3), dtype=np.float32)
        for h, hand in enumerate(result.hand_landmarks):
            for i, lm in enumerate(hand):
                out[h, i] = [lm.x, lm.y, lm.z]
        return torch.from_numpy(out)

    def close(self):
        self.landmarker.close()

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        input_dim = num_hands * tracking_points * tracking_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, classifier_hidden_dim),
            nn.ELU(),
            nn.Linear(classifier_hidden_dim, classifier_hidden_dim),
            nn.ELU(),
            nn.Linear(classifier_hidden_dim, latent_dim),
            nn.Tanh(),   # bounds latent to (−1, 1)
        )

    def forward(self, x):
        return self.net(x.flatten(start_dim=1))


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        output_dim = num_hands * tracking_points * tracking_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim, classifier_hidden_dim),
            nn.ELU(),
            nn.Linear(classifier_hidden_dim, classifier_hidden_dim),
            nn.ELU(),
            nn.Linear(classifier_hidden_dim, output_dim),
            # no final activation — bone offsets are unconstrained
        )

    def forward(self, z):
        return self.net(z)


class PoseAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

In [ ]:
_TOPOLOGY = [
    (0, 1), (1, 2), (2, 3), (3, 4),       # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),       # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16), # Ring
    (0, 17), (17, 18), (18, 19), (19, 20), # Pinky
]

def to_bone_offsets(tensor: torch.Tensor) -> torch.Tensor:
    """
    Convert absolute MediaPipe coords (N, 21, 3) to parent-relative joint offsets.

    Why this matters:
      Raw coords encode WHERE the hand is in the frame (wrist at ~0.5, 0.7).
      Moving the whole hand left/right shifts all 63 values equally, which a
      random or undertrained network cannot distinguish from a gesture change.

      Bone offsets encode the SHAPE of the hand only — wrist becomes (0,0,0)
      and each joint becomes its displacement from its parent. Translation and
      (roughly) scale invariant: only finger configuration is represented.
    """
    out = tensor.clone()
    for h in range(out.shape[0]):
        out[h, 0] = 0.0  # zero the wrist
        for parent, child in _TOPOLOGY:
            out[h, child] = tensor[h, child] - tensor[h, parent]
    return out

In [ ]:
pose_autoencoder = PoseAutoencoder().to(run_device)
hand_tracker     = HandTracker(model_path='hand_landmarker.task')
camera           = CameraWrapper(1)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class HandPoseDataset(Dataset):
    def __init__(self, poses):
        # poses: list of (num_hands, 21, 3) tensors in bone-offset space
        self.data = torch.stack(poses)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

## Data Collection

Move your hand(s) through all intended poses. Press **SPACE** to stop — valid detections are stored automatically in bone-offset space.

In [ ]:
# ── DATA COLLECTION ────────────────────────────────────────────────────────────
# Move your hand(s) through all intended poses. Press SPACE to stop.
from hand_visualizer import HandVisualizer

collected_poses = []
hand_tracker.reset()
start_time = time.time()
print('Collecting — move your hand through all poses. Press SPACE to stop...')

viz_collect = HandVisualizer('Collecting...', size=(400, 400))

while not keyboard.is_pressed('space'):
    frame     = camera()
    hand_pose = hand_tracker(frame, int((time.time() - start_time) * 1000))
    if hand_pose.size(0) != num_hands:
        continue
    offsets = to_bone_offsets(hand_pose)
    collected_poses.append(offsets.cpu())
    viz_collect.update(hand_pose)

viz_collect.close()

dataset = HandPoseDataset(collected_poses)
print(f'Collected {len(collected_poses)} samples  →  dataset ready')

## Region Detector

After training, the raw latent `z` (16-dim) is fed into an online k-means detector
instead of being argmax'd directly (that was the old design where latent_dim == num_action_classes).

Run the seeding cell below once before inference.

In [ ]:
class RegionDetector:
    """K-Means++ seeded online k-means with EMA centroid drift tracking."""

    def __init__(self, n_regions: int, latent_dim: int, momentum: float = 0.98):
        self.n_regions  = n_regions
        self.latent_dim = latent_dim
        self.momentum   = momentum
        self.centroids  = None

    def seed_from_batch(self, z_batch: torch.Tensor) -> None:
        z = z_batch.detach().cpu().float()
        N = z.shape[0]
        centers = [z[torch.randint(N, (1,)).item()]]
        for _ in range(self.n_regions - 1):
            dist_sq = torch.stack(
                [((z - c.unsqueeze(0)) ** 2).sum(dim=1) for c in centers], dim=1
            ).min(dim=1).values
            probs = dist_sq / (dist_sq.sum() + 1e-10)
            centers.append(z[torch.multinomial(probs, 1).item()])
        self.centroids = torch.stack(centers)

    def _sq_dists(self, z_flat):
        return ((self.centroids - z_flat.unsqueeze(0)) ** 2).sum(dim=1)

    def query(self, z: torch.Tensor) -> int:
        return int(self._sq_dists(z.detach().cpu().float().flatten()).argmin())

    def distances(self, z: torch.Tensor) -> torch.Tensor:
        return self._sq_dists(z.detach().cpu().float().flatten()).sqrt()


# ── Seed from the training dataset ────────────────────────────────────────────
region_detector = RegionDetector(n_regions=num_action_classes, latent_dim=latent_dim)

pose_autoencoder.eval()
with torch.no_grad():
    seed_batch = dataset.data.to(run_device)        # full training set
    _, z_seed  = pose_autoencoder(seed_batch)

region_detector.seed_from_batch(z_seed.cpu())
print(f'Region detector seeded.  Centroid norms: {region_detector.centroids.norm(dim=1).tolist()}')

In [ ]:
# ── AUTOENCODER TRAINING ───────────────────────────────────────────────────────
import matplotlib.pyplot as plt

num_epochs = 800
batch_size = 64
lr         = 1e-4

dataloader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)
optimizer   = torch.optim.Adam(pose_autoencoder.parameters(), lr=lr)
criterion   = nn.MSELoss()
epoch_losses = []

pose_autoencoder.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for batch in dataloader:
        batch        = batch.to(run_device)
        recon, _     = pose_autoencoder(batch)
        target       = batch.flatten(start_dim=1)
        loss         = criterion(recon, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss  += loss.item()
    avg = epoch_loss / len(dataloader)
    epoch_losses.append(avg)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{num_epochs}  loss={avg:.6f}')

print('Training complete.')

fig, ax = plt.subplots(figsize=(8, 3))
fig.patch.set_facecolor('#111111')
ax.set_facecolor('#111111')
ax.plot(epoch_losses, color='#3a8fff', linewidth=1.5)
ax.set_xlabel('Epoch', color='#aaaaaa')
ax.set_ylabel('MSE Loss', color='#aaaaaa')
ax.set_title('Autoencoder Training Loss', color='#dddddd')
ax.tick_params(colors='#aaaaaa')
for spine in ax.spines.values():
    spine.set_edgecolor('#444444')
plt.tight_layout()
plt.show()

## Inference

Bone-offset preprocessing applied.

In [ ]:
# ── INFERENCE ─────────────────────────────────────────────────────────────────
# Press 'q' to stop.
from hand_visualizer import HandVisualizer

_RECON_TREE = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
]

def offsets_to_abs(offsets, wrist_xy=(0.5, 0.5)):
    abs_pos = offsets.clone()
    for h in range(abs_pos.shape[0]):
        abs_pos[h, 0] = torch.tensor([wrist_xy[0], wrist_xy[1], 0.0])
        for parent, child in _RECON_TREE:
            abs_pos[h, child] = abs_pos[h, parent] + offsets[h, child]
    return abs_pos

# ──────────────────────────────────────────────────────────────────────────────

hand_tracker.reset()
start_time = time.time()
held_key   = None

viz_input  = HandVisualizer('Input (real hand)',       size=(400, 400))
viz_output = HandVisualizer('Output (reconstruction)', size=(400, 400))

pose_autoencoder.eval()
print("Running inference — press 'q' to stop\n")

with torch.no_grad():
    while not keyboard.is_pressed('q'):
        frame     = camera()
        hand_pose = hand_tracker(frame, int((time.time() - start_time) * 1000))
        if hand_pose.size(0) != num_hands:
            continue

        offsets       = to_bone_offsets(hand_pose)
        recon, z      = pose_autoencoder(offsets.to(run_device))

        region        = region_detector.query(z[0])
        pred_key      = action_labels[region] if region < len(action_labels) else '?'
        dists         = region_detector.distances(z[0])

        recon_offsets = recon.cpu().reshape(num_hands, tracking_points, tracking_dim)
        viz_input.update(hand_pose)
        viz_output.update(offsets_to_abs(recon_offsets))

        if press_buttons and pred_key != held_key:
            if held_key is not None:
                pyautogui.keyUp(held_key)
            pyautogui.keyDown(pred_key)
            held_key = pred_key

        max_d = dists.max().item()
        clear_output(wait=True)
        print(f'Region: [ {region} ]  →  [ {pred_key} ]\n')
        for k, d in enumerate(dists):
            d      = d.item()
            prox   = 1.0 - d / (max_d + 1e-8)
            filled = int(prox * 20)
            bar    = '█' * filled + '░' * (20 - filled)
            arrow  = '  ←' if k == region else ''
            lbl    = action_labels[k] if k < len(action_labels) else str(k)
            print(f'  {lbl}:  {bar}  d={d:.3f}{arrow}')

viz_input.close()
viz_output.close()

if held_key is not None:
    pyautogui.keyUp(held_key)

print('\nStopped.')